# MLP Architecture Tuning

Bayesian optimization (Optuna) over MLP architecture and training hyperparameters
using a single train/validation split. Since we're primarily tuning architecture
(layer sizes, depth, dropout, learning rate), a fixed temporal split is sufficient
and keeps iteration fast.

**Tunable parameters:**
- Number of hidden layers (1-4)
- Hidden layer widths
- Dropout rate
- Learning rate
- Batch size
- Weight decay

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import optuna
import torch

from src.features.gkx_registry import GKX_94
from src.ensemble import anchored_expanding_cv, MLPRanker

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# -- Load data (monthly modeling panel) --
run_tag = "run_5000_nonetf_1990"
gkx_path = f"../data/processed/{run_tag}/gkx_monthly.parquet"
daily_path = f"../data/processed/{run_tag}/final_dataset.parquet"

gkx = pd.read_parquet(gkx_path)
daily = pd.read_parquet(daily_path)

# Fix _x/_y suffixes from prior merges
daily = daily.drop(columns=[c for c in daily.columns if c.endswith("_x")], errors="ignore")
daily.columns = [c.replace("_y", "") for c in daily.columns]

daily["date"] = pd.to_datetime(daily["date"])
daily["month_end"] = daily["date"] + pd.offsets.MonthEnd(0)

# Month-end close per ticker
px_m = (
    daily.sort_values(["ticker", "date"])
    .groupby(["ticker", "month_end"], as_index=False)
    .tail(1)[["ticker", "month_end", "close"]]
)
px_m = px_m.sort_values(["ticker", "month_end"])
px_m["ret_eom_t+1"] = px_m.groupby("ticker")["close"].shift(-1) / px_m["close"] - 1

# SPY benchmark return
df_spy = px_m.loc[px_m["ticker"] == "SPY", ["month_end", "ret_eom_t+1"]].rename(
    columns={"ret_eom_t+1": "spy_ret_eom_t+1"}
)

# Macro features
macro_cols = [
    "DFF",
    "DGS10",
    "DGS2",
    "T10Y2Y",
    "CPIAUCSL",
    "CPILFESL",
    "PCEPI",
    "GDP",
    "GDPC1",
    "INDPRO",
    "UNRATE",
    "PAYEMS",
    "ICSA",
    "UMCSENT",
    "RSXFS",
    "VIXCLS",
    "DCOILWTICO",
    "M2SL",
    "TOTALSL",
]
macro_m = (
    daily.sort_values("date")
    .groupby("month_end", as_index=False)
    .tail(1)[["month_end"] + macro_cols]
    .drop_duplicates("month_end")
)

# Build modeling table
df = gkx.merge(
    px_m[["ticker", "month_end", "ret_eom_t+1"]],
    on=["ticker", "month_end"],
    how="left",
)
df = df.merge(df_spy, on="month_end", how="left")
df = df.merge(macro_m, on="month_end", how="left")

# Winsorize monthly cross-section of next-month returns
lo = df.groupby("month_end")["ret_eom_t+1"].transform(lambda x: x.quantile(0.01))
hi = df.groupby("month_end")["ret_eom_t+1"].transform(lambda x: x.quantile(0.99))
df["ret_eom_t+1"] = df["ret_eom_t+1"].clip(lower=lo, upper=hi)

# Features used for modeling
all_gkx_cols = [c for c in GKX_94 if c in df.columns]
feature_cols = all_gkx_cols + [c for c in macro_cols if c in df.columns]

# Filter to rows with valid target
model_df = df[["ticker", "month_end", "ret_eom_t+1"] + feature_cols].copy()
model_df = model_df[model_df["ret_eom_t+1"].notna()].reset_index(drop=True)
model_df = model_df.replace([np.inf, -np.inf], np.nan)

# Integrity checks
dup_keys = model_df.duplicated(["ticker", "month_end"]).sum()
monthly_n = model_df.groupby("month_end")["ticker"].nunique()
missing_by_feature = model_df[feature_cols].isna().mean().sort_values(ascending=False)
top_missing_features = missing_by_feature.head(20).rename("missing_ratio").to_frame()

model_df_checks = pd.DataFrame(
    {
        "metric": [
            "rows",
            "tickers",
            "months",
            "duplicate_(ticker,month_end)",
            "min_names_per_month",
            "median_names_per_month",
            "max_names_per_month",
        ],
        "value": [
            len(model_df),
            model_df["ticker"].nunique(),
            model_df["month_end"].nunique(),
            int(dup_keys),
            int(monthly_n.min()),
            float(monthly_n.median()),
            int(monthly_n.max()),
        ],
    }
)

print(model_df_checks.to_string(index=False))
print(f"Date range: {model_df['month_end'].min()} to {model_df['month_end'].max()}")
print(f"Total features: {len(feature_cols)}")
print("\nTop 10 most-missing features:")
print(top_missing_features.head(10).to_string())


In [ ]:
raw_gkx_cols = [c for c in GKX_94 if c in model_df.columns]
macro_feature_cols = [c for c in macro_cols if c in model_df.columns]

predictor_cols = (
    raw_gkx_cols
    + [f"{col}_csrank" for col in raw_gkx_cols]
    + [f"{col}_csnorm" for col in raw_gkx_cols]
    + macro_feature_cols
)

print(f"Raw GKX cols available: {len(raw_gkx_cols)}")
print(f"Macro cols available:   {len(macro_feature_cols)}")
print(f"Total predictor cols:   {len(predictor_cols)}")


In [ ]:
# Generate anchored-expanding CV folds.
# step_months is set to 72 to avoid cross-fold overlap between
# validation and holdout windows when val_months=36 and holdout_months=36.
cv_kwargs = dict(
    min_train_months=180,
    val_months=60,
    holdout_months=60,
    step_months=72,
    embargo_months=1,
)

folds = list(anchored_expanding_cv(model_df, date_col="month_end", **cv_kwargs))
print(f"Total folds: {len(folds)}\n")

unique_months = pd.Index(np.sort(model_df["month_end"].unique()))


def month_count(start, end):
    mask = (unique_months >= start) & (unique_months <= end)
    return int(mask.sum())


# Integrity checks: strict ordering + no within-fold overlap
cv_rows = []
for f in folds:
    train_mask = f["train_mask"]
    val_mask = f["val_mask"]
    holdout_mask = f["holdout_mask"]

    idx_train = np.flatnonzero(train_mask)
    idx_val = np.flatnonzero(val_mask)
    idx_holdout = np.flatnonzero(holdout_mask)

    train_val_overlap = int(np.intersect1d(idx_train, idx_val).size)
    train_holdout_overlap = int(np.intersect1d(idx_train, idx_holdout).size)
    val_holdout_overlap = int(np.intersect1d(idx_val, idx_holdout).size)

    strict_time_order = bool(
        f["train_end"] < f["val_start"] <= f["val_end"] < f["holdout_start"] <= f["holdout_end"]
    )

    cv_rows.append(
        {
            "fold": f["fold"],
            "train_end": f["train_end"].strftime("%Y-%m"),
            "val": f"{f['val_start'].strftime('%Y-%m')} -> {f['val_end'].strftime('%Y-%m')}",
            "holdout": f"{f['holdout_start'].strftime('%Y-%m')} -> {f['holdout_end'].strftime('%Y-%m')}",
            "train_months": month_count(unique_months.min(), f["train_end"]),
            "val_months": month_count(f["val_start"], f["val_end"]),
            "holdout_months": month_count(f["holdout_start"], f["holdout_end"]),
            "n_train_rows": int(train_mask.sum()),
            "n_val_rows": int(val_mask.sum()),
            "n_holdout_rows": int(holdout_mask.sum()),
            "train_val_overlap_rows": train_val_overlap,
            "train_holdout_overlap_rows": train_holdout_overlap,
            "val_holdout_overlap_rows": val_holdout_overlap,
            "strict_time_order": strict_time_order,
        }
    )

fold_df = pd.DataFrame(cv_rows)
print(fold_df.to_string(index=False))

if (
    fold_df[["train_val_overlap_rows", "train_holdout_overlap_rows", "val_holdout_overlap_rows"]]
    > 0
).any().any():
    raise ValueError("Fold leakage detected: overlap rows found within a fold")
if not fold_df["strict_time_order"].all():
    raise ValueError("Fold integrity failed: non-chronological split found")


In [ ]:
# Gantt chart of CV folds
fig, ax = plt.subplots(figsize=(14, max(4, len(folds) * 0.4)))

colors = {"train": "#2196F3", "val": "#FF9800", "holdout": "#4CAF50"}
dates_sorted = np.sort(model_df["month_end"].unique())
date_min, date_max = dates_sorted[0], dates_sorted[-1]

for f in folds:
    y = f["fold"]
    ax.barh(y, (f["train_end"] - date_min).days, left=0,
            height=0.6, color=colors["train"], alpha=0.7)
    ax.barh(y, (f["val_end"] - f["val_start"]).days,
            left=(f["val_start"] - date_min).days,
            height=0.6, color=colors["val"], alpha=0.7)
    ax.barh(y, (f["holdout_end"] - f["holdout_start"]).days,
            left=(f["holdout_start"] - date_min).days,
            height=0.6, color=colors["holdout"], alpha=0.7)

year_ticks = pd.date_range(date_min, date_max, freq="5YE")
ax.set_xticks([(t - date_min).days for t in year_ticks])
ax.set_xticklabels([t.strftime("%Y") for t in year_ticks])

ax.set_ylabel("Fold")
ax.set_xlabel("Date")
ax.set_title("Anchored-Expanding CV Folds")
ax.invert_yaxis()

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, alpha=0.7, label=l) for l, c in colors.items()]
ax.legend(handles=legend_elements, loc="upper right")
plt.tight_layout()
plt.show()

## Train / Validation / Test Split

Single temporal split with 1-month embargo between each window.

In [ ]:
# Temporal split: train < 2022, val 2022-2023, test 2024+
TRAIN_END = "2021-12-31"
VAL_START = "2022-02-01"   # 1-month embargo
VAL_END = "2023-12-31"
TEST_START = "2024-02-01"  # 1-month embargo

train_mask = model_df["month_end"] <= TRAIN_END
val_mask = (model_df["month_end"] >= VAL_START) & (model_df["month_end"] <= VAL_END)
test_mask = model_df["month_end"] >= TEST_START

X_train = model_df.loc[train_mask, feature_cols]
y_train = model_df.loc[train_mask, "target_excess_t+1"]
X_val = model_df.loc[val_mask, feature_cols]
y_val = model_df.loc[val_mask, "target_excess_t+1"]
X_test = model_df.loc[test_mask, feature_cols]
y_test = model_df.loc[test_mask, "target_excess_t+1"]
me_test = model_df.loc[test_mask, "month_end"]

print(f"Train: {train_mask.sum():,} rows ({model_df.loc[train_mask, 'month_end'].nunique()} months)")
print(f"Val:   {val_mask.sum():,} rows ({model_df.loc[val_mask, 'month_end'].nunique()} months)")
print(f"Test:  {test_mask.sum():,} rows ({model_df.loc[test_mask, 'month_end'].nunique()} months)")

## Helpers

In [ ]:
def ic_scorer(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Spearman rank IC between actual excess returns and predicted scores."""
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() < 10:
        return 0.0
    corr, _ = spearmanr(y_true[mask], y_pred[mask])
    return float(corr) if np.isfinite(corr) else 0.0


def build_hidden_dims(trial, n_layers):
    """Suggest hidden layer widths. Layers taper by default."""
    dims = []
    prev = len(feature_cols)  # input dim as upper reference
    for i in range(n_layers):
        # Each layer can be 32-512, but suggest in powers-of-2 neighborhood
        hi = min(512, max(64, prev))
        dim = trial.suggest_int(f"hidden_{i}", 32, hi, step=32)
        dims.append(dim)
        prev = dim
    return dims

---
## Bayesian Optimization (Optuna)

In [ ]:
N_TRIALS = 40

def objective(trial):
    n_layers = trial.suggest_int("n_layers", 1, 4)
    hidden_dims = build_hidden_dims(trial, n_layers)
    dropout = trial.suggest_float("dropout", 0.1, 0.5, step=0.05)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [1024, 2048, 4096, 8192])
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)

    ranker = MLPRanker(
        hidden_dims=hidden_dims,
        dropout=dropout,
        lr=lr,
        epochs=100,  # early stopping handles actual duration
        batch_size=batch_size,
        patience=10,
    )

    # Inject weight decay into Adam optimizer by monkey-patching fit
    # (MLPRanker uses Adam; we store weight_decay for the custom fit below)
    ranker._weight_decay = weight_decay
    _orig_fit = ranker.fit

    def _fit_with_wd(X, y, X_val=None, y_val=None):
        import torch.nn as nn
        from torch.utils.data import DataLoader, TensorDataset

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        X_arr = ranker._impute_and_scale(X, fit=True)
        y_arr = y.values.astype(np.float32).reshape(-1, 1)
        y_arr = np.nan_to_num(y_arr, nan=0.0)

        train_ds = TensorDataset(torch.from_numpy(X_arr), torch.from_numpy(y_arr))
        train_dl = DataLoader(train_ds, batch_size=ranker.batch_size, shuffle=True)

        has_val = X_val is not None and y_val is not None
        if has_val:
            X_val_arr = ranker._impute_and_scale(X_val)
            y_val_arr = y_val.values.astype(np.float32).reshape(-1, 1)
            y_val_arr = np.nan_to_num(y_val_arr, nan=0.0)
            val_X_t = torch.from_numpy(X_val_arr).to(device)
            val_y_t = torch.from_numpy(y_val_arr).to(device)

        ranker.model_ = ranker._build_model(X_arr.shape[1]).to(device)
        optimizer = torch.optim.Adam(
            ranker.model_.parameters(), lr=ranker.lr, weight_decay=weight_decay
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=ranker.epochs)
        criterion = nn.MSELoss()

        best_val_loss = float("inf")
        best_state = None
        patience_counter = 0

        for epoch in range(ranker.epochs):
            ranker.model_.train()
            for xb, yb in train_dl:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                loss = criterion(ranker.model_(xb), yb)
                loss.backward()
                optimizer.step()
            scheduler.step()

            if has_val:
                ranker.model_.eval()
                with torch.no_grad():
                    val_loss = criterion(ranker.model_(val_X_t), val_y_t).item()
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_state = {
                        k: v.cpu().clone() for k, v in ranker.model_.state_dict().items()
                    }
                    patience_counter = 0
                else:
                    patience_counter += 1
                    if patience_counter >= ranker.patience:
                        break

                # Optuna pruning: report val loss at each epoch
                trial.report(val_loss, epoch)
                if trial.should_prune():
                    raise optuna.TrialPruned()

        if best_state is not None:
            ranker.model_.load_state_dict(best_state)
        ranker.model_.eval()
        ranker.model_.to("cpu")
        return ranker

    _fit_with_wd(X_train, y_train, X_val, y_val)

    preds = ranker.predict(X_val)
    ic = ic_scorer(y_val.values, preds)

    # Store architecture info
    trial.set_user_attr("hidden_dims", hidden_dims)
    trial.set_user_attr("val_ic", ic)

    return ic


optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10),
    study_name="mlp_tuning",
)

# Seed with the default MLPRanker architecture
study.enqueue_trial({
    "n_layers": 3,
    "hidden_0": 256, "hidden_1": 128, "hidden_2": 64,
    "dropout": 0.3,
    "lr": 1e-3,
    "batch_size": 4096,
    "weight_decay": 1e-5,
})

print(f"Running {N_TRIALS} trials...")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest trial #{study.best_trial.number}:")
print(f"  Val IC: {study.best_value:.4f}")
print(f"  Params: {study.best_params}")
print(f"  Architecture: {study.best_trial.user_attrs['hidden_dims']}")

## Analysis

In [ ]:
# Top 10 trials
trials_df = study.trials_dataframe()
trials_df = trials_df[trials_df["state"] == "COMPLETE"].sort_values("value", ascending=False)

display_cols = ["number", "value", "duration"] + [c for c in trials_df.columns if c.startswith("params_")]
print("Top 10 trials:")
print(trials_df[display_cols].head(10).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Optimization history
ax = axes[0]
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
trial_nums = [t.number for t in completed]
trial_vals = [t.value for t in completed]
ax.scatter(trial_nums, trial_vals, alpha=0.5, s=30, color="#2196F3")
running_best = np.maximum.accumulate(trial_vals)
ax.plot(trial_nums, running_best, color="red", linewidth=2, label="Best so far")
ax.set_xlabel("Trial")
ax.set_ylabel("Val IC")
ax.set_title("Optimization History")
ax.legend()

# Hyperparameter importance
ax = axes[1]
try:
    importance = optuna.importance.get_param_importances(study)
    ax.barh(list(importance.keys()), list(importance.values()), color="#FF9800")
    ax.set_xlabel("Importance")
    ax.set_title("Hyperparameter Importance")
except Exception:
    ax.text(0.5, 0.5, "Need more completed trials",
            ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
plt.show()

In [ ]:
# Scatter: IC vs key continuous hyperparameters
cont_params = ["params_dropout", "params_lr", "params_weight_decay", "params_n_layers"]
avail = [p for p in cont_params if p in trials_df.columns]

fig, axes = plt.subplots(1, len(avail), figsize=(4 * len(avail), 3.5))
if len(avail) == 1:
    axes = [axes]

for ax, col in zip(axes, avail):
    ax.scatter(trials_df[col], trials_df["value"], alpha=0.6, s=40)
    label = col.replace("params_", "")
    ax.set_xlabel(label)
    ax.set_ylabel("Val IC")
    ax.set_title(label)
    if "lr" in col or "weight_decay" in col:
        ax.set_xscale("log")

plt.suptitle("IC vs Hyperparameters", y=1.02)
plt.tight_layout()
plt.show()

## Refit Best & Evaluate on Test Set

In [ ]:
# Refit the best architecture on train, early-stop on val, evaluate on test
best_trial = study.best_trial
best_hidden = best_trial.user_attrs["hidden_dims"]
bp = best_trial.params

print(f"Refitting best architecture: {best_hidden}")
print(f"  dropout={bp['dropout']}, lr={bp['lr']:.5f}, "
      f"batch_size={bp['batch_size']}, weight_decay={bp['weight_decay']:.6f}\n")

best_ranker = MLPRanker(
    hidden_dims=best_hidden,
    dropout=bp["dropout"],
    lr=bp["lr"],
    epochs=150,    # allow more epochs for final fit
    batch_size=bp["batch_size"],
    patience=15,
)
best_ranker.fit(X_train, y_train, X_val, y_val)

# Evaluate on val and test
val_preds = best_ranker.predict(X_val)
test_preds = best_ranker.predict(X_test)

val_ic = ic_scorer(y_val.values, val_preds)
test_ic = ic_scorer(y_test.values, test_preds)

print(f"Val  IC: {val_ic:.4f}")
print(f"Test IC: {test_ic:.4f}")

In [ ]:
# Decile analysis on test set
test_df = pd.DataFrame({
    "pred": test_preds,
    "ret": y_test.values,
    "month_end": me_test.values,
})
test_df["pred_decile"] = test_df.groupby("month_end")["pred"].transform(
    lambda x: pd.qcut(x, 10, labels=False, duplicates="drop")
)

decile_stats = test_df.groupby("pred_decile")["ret"].agg(["mean", "std", "count"])
decile_stats["mean_annualized"] = decile_stats["mean"] * 12

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(decile_stats.index, decile_stats["mean"] * 100,
       color=["#f44336" if i < 3 else "#4CAF50" if i > 6 else "#90A4AE"
              for i in decile_stats.index], alpha=0.8)
ax.set_xlabel("Predicted Decile")
ax.set_ylabel("Mean Monthly Excess Return (%)")
ax.set_title("Test Set: Excess Return by Predicted Decile")
ax.axhline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

spread = decile_stats["mean"].iloc[-1] - decile_stats["mean"].iloc[0]
print(f"\nTop-bottom decile spread: {spread*100:.2f}% monthly ({spread*1200:.1f}% annualized)")
print(f"\n{decile_stats}")

## Save Best Architecture

In [ ]:
best_mlp_config = {
    "hidden_dims": best_hidden,
    "dropout": bp["dropout"],
    "lr": bp["lr"],
    "batch_size": bp["batch_size"],
    "weight_decay": bp["weight_decay"],
    "val_ic": val_ic,
    "test_ic": test_ic,
}

out_path = f"../data/processed/{run_tag}/best_mlp_config.json"
with open(out_path, "w") as f:
    json.dump(best_mlp_config, f, indent=2)

print(f"Saved to {out_path}")
print(f"\nUsage:")
print(f"  from src.ensemble import MLPRanker")
print(f"  ranker = MLPRanker(hidden_dims={best_hidden}, dropout={bp['dropout']}, "
      f"lr={bp['lr']:.5f}, batch_size={bp['batch_size']})")